In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import torch

from helmholtz_eos_net.test import load_model_from_folder


msize=12
mwidth = 2
fsize=16
alpha=0.3
lsize = 2
framewidth = 3
markeredgewidth=3
markeredgewidth_no=10

print('Imports successful!')

In [ ]:

# the base model in the paper
model_folder = "ray_results/d29fd_00000/"

# the water model in the paper
model_folder = "trained_models/transport_joint_water_res_08c6/" 


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config = load_model_from_folder(model_folder, device)
model.eval()

In [ ]:
# Single VLE reference for all three figures: the NIST WebBook saturation table
# (IAPWS-95). Carries p_sat, rho_liq, rho_vap, dHvap and the SAFT parameters.
vle_df   = pd.read_csv('water_nist/water_nist_vle.csv')

# Homogeneous p-rho-T backdrop — the actual residual training split.
state_df = pd.read_csv('water_data/data_res_full_train.csv')


# Temperature grid — change T_step to adjust resolution
T_step = 10.0
#T_step = 3.0
#T_step = 1.0
T_min  = vle_df['temperature_K'].min() + 5 #T_step
#T_min = 400.0
T_max  = vle_df['temperature_K'].max() - 20 # T_step
temps  = np.arange(T_min, T_max, T_step)
print(f'Evaluating VLE at {len(temps)} temperatures ({T_min:.1f}–{T_max:.1f} K, step={T_step} K)')

In [ ]:
from feos.pcsaft import PcSaftParameters, PcSaftRecord, Identifier, PureRecord
from feos.eos import EquationOfState, State, PhaseDiagram
from feos.eos import Contributions
from si_units import * 

eos_csv = "SI_pcp-saft_parameters.csv"

eos_keys = ["m","sigma","epsilon_k","kappa_ab","epsilon_k_ab","mu"]
embedding_keys = ["molarweight","m","sigma","epsilon_k","kappa_ab","epsilon_k_ab","mu"]

embedding_paras = pd.read_csv( eos_csv, sep="\t")
embedding_paras = embedding_paras.fillna( 0.0 )

print( np.where( embedding_paras.common_name == "water") )
sub = embedding_paras.iloc[156]


M = sub["molarweight"]*(GRAM/MOL)
m = sub["m"]

identifier = Identifier( cas="000-00-0", name="dummy" )
saftrec = PcSaftRecord( m=sub["m"], sigma=sub["sigma"], epsilon_k=sub["epsilon_k"],
                      kappa_ab=sub["kappa_ab"],epsilon_k_ab=sub["epsilon_k_ab"],mu=sub["mu"],
                       na=sub["na"], nb=sub["nb"] )

pr = PureRecord(identifier=identifier,model_record=saftrec,molarweight=sub["molarweight"])

saft_paras = PcSaftParameters.from_records( [pr], np.array([[0.0]] ) )

eos = EquationOfState.pcsaft(saft_paras)

saft_vle = PhaseDiagram.pure(eos, min_temperature=273.2*KELVIN , npoints=100)

temperature_saft = saft_vle.vapor.temperature / KELVIN
pressure_saft = saft_vle.vapor.pressure / PASCAL
rho_vap_saft = saft_vle.vapor.density / MOL * METER**3
rho_liq_saft = saft_vle.liquid.density / MOL * METER**3
h_vap = saft_vle.vapor.molar_enthalpy(Contributions.Residual)
h_liq = saft_vle.liquid.molar_enthalpy(Contributions.Residual)
dh_vap_saft = (h_vap - h_liq) /JOULE *MOL


dh_vap_saft


In [ ]:
# Interpolate initial guesses from the NIST reference
vle_sorted = vle_df.sort_values('temperature_K')
f_gas = interp1d(vle_sorted['temperature_K'], vle_sorted['molar_density_vapor_mol_m3'],  kind='linear', fill_value='extrapolate')
f_liq = interp1d(vle_sorted['temperature_K'], vle_sorted['molar_density_liquid_mol_m3'], kind='linear', fill_value='extrapolate')

rho_gas_init = torch.tensor( 1* f_gas(temps)[:, None], dtype=torch.float32)
rho_liq_init = torch.tensor( 1* f_liq(temps)[:, None], dtype=torch.float32)
T_tensor     = torch.tensor(temps[:, None], dtype=torch.float32)

# Substance parameters (same for all water rows)
branch_input = torch.tensor(
    np.tile(vle_df[model.substance_params.parameter_columns].iloc[0].values, (len(temps), 1)),
    dtype=torch.float32
)

print(f'T_tensor:     {T_tensor.shape}')
print(f'rho_gas_init: {rho_gas_init.shape}')
print(f'rho_liq_init: {rho_liq_init.shape}')
print(f'branch_input: {branch_input.shape}')

In [ ]:
branch_input

In [ ]:
if False:
    branch_input[:,0] = branch_input[:,0] + 0.01
    branch_input[:,1] = branch_input[:,1] + 0.01
    branch_input[:,2] = branch_input[:,2] + 0.01

    branch_input[:,0] = branch_input[:,0] - 0.03
    branch_input[:,1] = branch_input[:,1] - 0.02
    branch_input[:,2] = branch_input[:,2] - 0.01

# Run batch VLE flash
with torch.enable_grad():
    p_sat_pred, mu_sat_pred, rho_gas_vle, rho_liq_vle, converged = model.find_vle_flash_batch(
        branch_input, T_tensor, rho_gas_init, rho_liq_init,
        n_steps=40, tol=1e-6, alpha=0.5,
        # determine_rho_liq=0*torch.ones(T_tensor.shape[0], 1)  # freeze liquid at rho_liq_init
    )

rho_gas_vle = rho_gas_vle.detach().numpy().squeeze()
rho_liq_vle = rho_liq_vle.detach().numpy().squeeze()
converged   = converged.numpy()
p_sat_pred  = p_sat_pred.detach().numpy().squeeze()

temperature = T_out.numpy()

print(f'Converged: {converged.sum()}/{len(converged)}')

In [ ]:
# ── Stepped VLE curve: predictor-corrector from T_max downward ──────────────
p_water = branch_input[0:1]       # (1, 8) — single substance for all steps
T_1d    = T_tensor.squeeze()      # (n_T,) — any order, sorted internally

# Initial guess at the highest temperature in temps (NIST reference).
# Only these two scalars are consumed — find_vle_curve marches down from here.
rho_l0 = float(rho_liq_init[-1])
rho_g0 = float(rho_gas_init[-1])

with torch.enable_grad():
    if False:
        p_sat_out, mu_sat_pred, rho_gas_out, rho_liq_out, converged_out = model.find_vle_flash_batch(
            branch_input, T_tensor, rho_gas_init, rho_liq_init,
            n_steps=40, tol=1e-6, alpha=0.5,
            # determine_rho_liq=0*torch.ones(T_tensor.shape[0], 1)  # freeze liquid at rho_liq_init
        )        
        T_out = T_tensor
    else:
        T_out, p_sat_out, p_sat_cc_out, rho_liq_out, rho_gas_out, dHvap_out, converged_out = \
            model.find_vle_curve(p_water, T_1d, rho_l0, rho_g0, n_steps=40)
            

temperature = T_out.numpy()
rho_gas_vle = rho_gas_out.detach().numpy()
rho_liq_vle = rho_liq_out.detach().numpy()
p_sat_pred  = p_sat_out.detach().numpy()
p_sat_cc    = p_sat_cc_out.detach().numpy()
delta_H_vap = dHvap_out.detach().numpy()
converged   = converged_out.detach().numpy()

print(f'Converged: {converged.sum()}/{len(converged)}')


In [ ]:
plt.plot( state_df.molar_density_mol_m3, state_df.temperature_K, ".y" )

plt.plot( vle_df.molar_density_liquid_mol_m3, vle_df.temperature_K, "-g" )
plt.plot( vle_df.molar_density_vapor_mol_m3, vle_df.temperature_K, "-g" )

plt.plot(rho_gas_vle, temperature, ".-m" )
plt.plot(rho_liq_vle, temperature, ".-m" )


In [ ]:

plt.plot( state_df.molar_density_mol_m3, state_df.temperature_K, ".y" )

plt.plot(rho_gas_vle, temperature, ".-m" )
plt.plot(rho_liq_vle, temperature, ".-m" )

In [ ]:
plt.plot( vle_df.temperature_K, vle_df.pressure_sat_Pa, ".y")

plt.plot( temperature, p_sat_pred, ".-m" )


In [ ]:


plt.plot( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa) , ".y")

plt.plot( temperature, np.log(p_sat_cc), ".-g" )

plt.plot( temperature, np.log(p_sat_pred), ".m" )

p = converged.squeeze() == False
plt.plot( temperature[p], np.log(p_sat_pred[p]), "xm" )


In [ ]:

temperature_grid, _ = np.meshgrid(temperature, temperature)
pressure_grid, _ = np.meshgrid(p_sat_pred, p_sat_pred)
delta_H_vap_grid, _ = np.meshgrid(delta_H_vap, delta_H_vap)



temperature_diag = np.diag(temperature_grid)[:, np.newaxis].repeat(temperature_grid.shape[0], axis=1)

t_diff = temperature_grid - temperature_diag
t_diff = np.abs(t_diff)
#weights = 1 - t_diff / t_diff.sum(axis=1, keepdims=True)
#weights[t_diff > 100] = 0
weights = 1/(1 + np.exp(t_diff/40))
weights = weights / weights.sum(axis=1, keepdims=True)

p_sat_ccc = pressure_grid * np.exp(delta_H_vap_grid / 8.314 * (1.0 / temperature_grid - 1.0 / temperature_diag))

p_sat_ccc = p_sat_ccc * weights
p_sat_ccc = p_sat_ccc.sum(axis=1)

plt.plot( temperature_saft, np.log(pressure_saft), "k--", label="PCP-SAFT")
plt.plot( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa) ,  ".", color="limegreen",markersize=msize, label="NIST / IAPWS-95")

#plt.plot( temperature, np.log(p_sat_ccc), ".-g" )

f_p_vap = interp1d( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa), fill_value="extrapolate" )

ref = np.exp(f_p_vap(temperature))
error = np.mean( np.abs( p_sat_pred -  ref) / ref )

plt.plot( temperature, np.log(p_sat_pred), "m.-",markersize=msize , label=f"water ({error*100:.2f}%)" )

ref = np.exp(f_p_vap(temperature_saft))
pcs_error = np.mean( np.abs( pressure_saft -  ref) / ref )

plt.plot( temperature_saft, np.log(pressure_saft), "k--")

plt.xlabel(r"temperature / K",fontsize=fsize)
plt.ylabel(r"$ln(p ~/~ Pa)$",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)


plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  


plt.savefig(model_folder+"VLE_water_pressure.png",bbox_inches='tight')
plt.savefig(model_folder+"VLE_water_pressure.pdf",bbox_inches='tight')

print(f"p_sat  model {error*100:.2f}%   PC-SAFT {pcs_error*100:.2f}%")


In [ ]:
plt.plot( temperature_saft, dh_vap_saft/1000, "k--", label="PCP-SAFT")

plt.plot( vle_df.temperature_K, vle_df.molar_vaporization_enthalpy_J_mol/1000, ".", color="limegreen",markersize=msize, label="NIST / IAPWS-95" )

f_dh_vap = interp1d( vle_df.temperature_K, vle_df.molar_vaporization_enthalpy_J_mol)

ref = f_dh_vap(temperature)
error = np.mean( np.abs( delta_H_vap -  ref) / ref )


plt.plot( temperature, delta_H_vap/1000, "m.-",markersize=msize , label=f"water ({error*100:.2f}%)" )
# plt.plot( temperature[p], delta_H_vap[p]/1000, "xm" )

plt.plot( temperature_saft, dh_vap_saft/1000, "k--")

plt.xlabel(r"temperature / K",fontsize=fsize)
plt.ylabel(r"$\Delta h_{\text{vap}}~/~kJ/mol$",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  

plt.savefig(model_folder+"VLE_water_enthalpy.png",bbox_inches='tight')
plt.savefig(model_folder+"VLE_water_enthalpy.pdf",bbox_inches='tight')

print(f"dHvap  model {error*100:.2f}%")

In [ ]:

plt.plot( rho_liq_saft/1000, temperature_saft, "k--", label="PC-SAFT")
plt.plot( state_df.molar_density_mol_m3/1000, state_df.temperature_K, ".", color="limegreen", label="experimental train" )

plt.plot(vle_df.molar_density_vapor_mol_m3/1000, vle_df.temperature_K, ".", color="limegreen",markersize=msize )
plt.plot(vle_df.molar_density_liquid_mol_m3/1000, vle_df.temperature_K, ".", color="limegreen",markersize=msize, label="NIST / IAPWS-95" )

f_rho_vap = interp1d( vle_df.temperature_K, vle_df.molar_density_vapor_mol_m3)
f_rho_liq = interp1d( vle_df.temperature_K, vle_df.molar_density_liquid_mol_m3)

ref = f_rho_vap(temperature)
error_vap = np.mean( np.abs( rho_gas_vle -  ref) / ref )

ref = f_rho_liq(temperature)
error_liq = np.mean( np.abs( rho_liq_vle -  ref) / ref )

plt.plot(rho_gas_vle/1000, temperature, ".-m",markersize=msize )
plt.plot(rho_liq_vle/1000, temperature, ".-m",markersize=msize , label=f"water (L {error_liq*100:.2f}%, V {error_vap*100:.2f}%)" )

plt.plot( rho_vap_saft/1000, temperature_saft, "k--")
plt.plot( rho_liq_saft/1000, temperature_saft, "k--")


plt.xlabel(r"$\rho~/~kmol/m^3$",fontsize=fsize)
plt.ylabel(r"temperature / K",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  

#plt.xlabel(r"$-s_\text{res}~/~R$",fontsize=fsize)
#plt.ylabel(r"$\ln(\eta~/~Pas)$",fontsize=fsize)

plt.savefig(model_folder+"VLE_water_density.png",bbox_inches='tight')
plt.savefig(model_folder+"VLE_water_density.pdf",bbox_inches='tight')

error_liq, error_vap

In [ ]:
plt.plot( temperature, rho_gas_vle, "m." )
plt.plot( temperature, np.ones(temperature.shape)*np.exp(3), "k-" )

In [ ]:
# temperature, delta_H_vap, rho_gas_vle

dvvap = 1/rho_gas_vle - 1/rho_liq_vle

dpdt = delta_H_vap/dvvap

plt.plot(  np.log( np.sort(dpdt) ) )

In [ ]:

dvvap = 1/rho_gas_vle - 1/rho_liq_vle

dpdt = delta_H_vap/dvvap/p_sat_pred

plt.plot(  np.log( dpdt ) )